# Certify a candidate with both sides of an LP

AuthorityEffect: None.

> **Claim-typing legend**
> - **[Pinned]**: reproduced from verified checked-in Grammar 000 fixtures.
> - **[Executed]**: verified by deterministic Python execution in this notebook.
> - **[Declared]**: pedagogical convention or beginner framing.
> - **[Open]**: questions left for future exploration.


## Start here

A proposer supplies a vertex potential, residual upper bounds, and a dual cycle. The checker must validate feasibility before comparing objectives. Equal displayed numbers alone have no such force.

## Exact model

min sum(w_e u_e), u_e >= |c_e-D alpha|; max y.c, D^T y=0, |y_e|<=w_e.

All numerical fixtures use integers or Fraction. A passing run means the named finite checks passed.


In [ ]:
import importlib.util
import json
import sys
from fractions import Fraction
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "omega_engine_education/scripts/build_system_integration.py").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from omega_engine_education.scripts.system_integration_support import (
    expect_error,
)


def load_source(name, relative):
    spec = importlib.util.spec_from_file_location(name, ROOT / relative)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

## Worked computation

Run the cell, then compare the output with the exact model above.


In [ ]:
lp = load_source(
    "lp_lesson",
    "omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/checker.py",
)
spec = {
    "matrices": {"d0": [[-1, 1, 0], [0, -1, 1], [1, 0, -1]]},
    "target_cochain": {"values": ["1/3"] * 3},
    "carrier": {"edges": [{"weight": "1"}] * 3},
}
witness = json.loads(
    (
        ROOT
        / "omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/witness_valid.json"
    ).read_text(encoding="utf-8")
)
report = lp.verify_lp_certificate(spec, witness)
assert report.admitted and report.is_exact_zero_gap
assert report.primal_objective == report.dual_objective == "1"
# Independent lower bound: |r0|+|r1|+|r2| >= |r0+r1+r2| = 1.
D = [[Fraction(x) for x in row] for row in spec["matrices"]["d0"]]
assert [sum(row[col] for row in D) for col in range(3)] == [0, 0, 0]
assert sum(Fraction(x) for x in spec["target_cochain"]["values"]) == 1
print("Primal and dual objectives are both one after feasibility checks.")

## Exercise and adversarial check

Corrupt one primal bound, break dual closure, and give a feasible dual with a positive gap.

Predict the result before running the reference check below. Each assertion checks a specific value or failure mode.


In [ ]:
from copy import deepcopy

cases = [
    ("primal", "REFUSED_PRIMAL_INFEASIBLE"),
    ("closure", "REFUSED_DUAL_INFEASIBLE"),
    ("gap", "REFUSED_NONZERO_DUALITY_GAP"),
]
for name, expected in cases:
    bad = deepcopy(witness)
    if name == "primal":
        bad["primal_candidate"]["primal_upper_bound_u"][0] = "0"
    if name == "closure":
        bad["dual_witness"]["w"] = ["1", "1", "0"]
    if name == "gap":
        bad["dual_witness"]["w"] = ["0", "0", "0"]
    actual = lp.verify_lp_certificate(spec, bad)
    assert not actual.admitted and actual.status == expected
expect_error(ValueError, lambda: lp.parse_rational(True, "candidate"))
print("Three independent false certificates are refused for different reasons.")

## Evidence boundary

This verifies a supplied certificate for a three-edge rational LP. It does not run an optimizer or classify all LPs.

## Source route

- [omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/checker.py](../../../../../omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/checker.py)
- [omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/witness_valid.json](../../../../../omega_engine_education/curriculum/the_symbol_game/demos/demo_4/dossier_2_exact_lp_certificate/witness_valid.json)

Return to the [intermediate integration route](../verified_route.md) and [curriculum overview](../curriculum_25_200.md).
